**Importar Bibliotecas**

In [2]:
import pandas as pd
import re
from faker import Faker
from datetime import datetime, date, timedelta
import inspect as inspe
import sys
import numpy as np
import openpyxl
import xlsxwriter as xlw
import unicodedata
import time
import shutil
import openpyxl.styles
from openpyxl.styles import Alignment
# Initialize Faker here so it's available for subsequent cells
fake = Faker('pt_BR')

**Cógios Auxiliares**

In [3]:
# Alinhas as datas da semana na coluna "SEMANAS"
Dias_semana = {
    'Monday': 'Segunda-feira',
    'Tuesday': 'Terça-feira',
    'Wednesday': 'Quarta-feira',
    'Thursday': 'Quinta-feira',
    'Friday': 'Sexta-feira',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}

filiais = [1, 12, 13, 15]

# Selecionar a situação
# Se estiver antes do vencimento, acionar o  status "A vencer", pós o vencimento, "VENCIDO"
# Se tiver valor devendo, ficará reconhecido como 'NEGOCIAÇÃO'
situacao_pgto = [
    'PAGO', 'CANCELADO', 'VENCIDO', 'A VENCER', 'NEGOCIAÇÃO'
]

fornecedores = [
    'Apex Industrial',
    'ITAÚ',
    'Nexus Equipamentos',
    'Beta Distribuidora',
    'INTER',
    'Delta Importações',
    'Gamma Tech Solutions',
    'Delta Importações'
    ] # Talvez pego da base de fornecedores

tipos_despesas = [
    'OPERACIONAIS', 'ADMINISTRATIVAS', 'FINANCEIRAS', 'FISCAIS', 'OUTRAS DESPESAS'
]  # Ativamente inseridas automaticamente

tipos_pagamentos = ['BOLETO', 'TRANSFERÊNCIA', 'PIX', 'DINHEIRO', 'CARTÃO'] # Selecionar por sigla

grupos_pgto = [
            'PAGAMENTOS FORNECEDORES', 'PAGAMENTO DE PJ', 'PAGAMENTO DE REEMBOLSOS', 'PAGAMENTOS IMPOSTOS E ENCARGOS SOCIAIS',
        'PAGAMENTOS SALÁRIOS/ BENEF / CONVÊNIOS', 'PAGAMENTOS ALUGUEIS/ CONDOMÍNIO / ENERGIA ELÉTRICA',
        'PAGAMENTOS TELEFONIA', 'CARTÃO DE CRÉDITO', 'CAPITAL DE GIRO', 'PRO LABORE', 'DISTRIBUIÇÃO DE LUCROS',
        'CONTAS DE CONSUMO', 'TARIFAS / JUROS (-)', 'OUTROS CRÉD. (REND/TED DEV./APORTE/RESGATE)',
        'INVESTIMENTO - ROBERT BOSH', 'INVESTIMENTO - FEDERAÇÃO'
] # Quero que sejam inseridos por reconhecimento, mas tenho que entender

# Área de reconhecimento do 'grupo_pgto' para inserir na coluna 'TIPO DE DESPESA' para o PowerBI
reconhecimento_despesas = (
    {
        'OPERACIONAIS': [
            'PAGAMENTOS FORNECEDORES',
            'PAGAMENTOS ALUGUEIS/ CONDOMÍNIO / ENERGIA ELÉTRICA',
            'CONTAS DE CONSUMO',
            'PAGAMENTOS TELEFONIA',
        ],
        'ADMINISTRATIVAS': [
            'PAGAMENTOS SALÁRIOS/ BENEF / CONVÊNIOS', 'PRO LABORE',
            'PAGAMENTO DE PJ'
        ],
        'FINANCEIRAS':
        ['TARIFAS / JUROS (-)', 'CAPITAL DE GIRO', 'CARTÃO DE CRÉDITO'],
        'OUTRAS DESPESAS': [
            'PAGAMENTO DE REEMBOLSOS', 'DISTRIBUIÇÃO DE LUCROS',
            'INVESTIMENTO - ROBERT BOSH', 'INVESTIMENTO - FEDERAÇÃO'
        ],
        'FISCAIS': ['PAGAMENTOS IMPOSTOS E ENCARGOS SOCIAIS']
    }
)  # Após colocar os dados na lista de despesas, estas informações serão reconhecidas e inseridas na última coluna chamada 'TIPO DE DESPESA'


bancos = {
    'C. CORRENTE – Matriz – 341 – ITAÚ LOGSERV RS – 4423 | 1320-9': '1',
    'C. CORRENTE – Filial – 341 – ITAÚ LOGSERV SP – 1182 | 5501-4': '2',
    'C. CORRENTE – Filial – 341 – ITAÚ LOGSERV PR – 2290 | 7742-1': '3',
    'C. CORRENTE – Filial – 341 – ITAÚ LOGSERV MG – 3301 | 9910-3': '4',
    'C. CORRENTE – Matriz – 237 – BRADESCO MEGASTORE SP – 5120 | 6674-0': '5',
    'C. CORRENTE – Filial – 237 – BRADESCO MEGASTORE RJ – 3344 | 2001-8': '6',
    'C. CORRENTE – Filial – 237 – BRADESCO MEGASTORE RS – 7811 | 4412-5': '7',
    'C. CORRENTE – Filial – 237 – BRADESCO MEGASTORE BA – 5602 | 7010-6': '8',
    'C. CORRENTE – Matriz – 033 – SANTANDER TECHPLUS MG – 6400 | 3528-4': '9',
    'C. CORRENTE – Filial – 033 – SANTANDER TECHPLUS SP – 2988 | 9400-1': '10',
    'C. CORRENTE – Filial – 033 – SANTANDER TECHPLUS SC – 1855 | 1082-7': '11',
    'C. CORRENTE – Filial – 033 – SANTANDER TECHPLUS PR – 7720 | 6509-9': '12',
    'C. CORRENTE – Matriz – 001 – BB AGROFORT MT – 1001 | 7712-0': '13',
    'C. CORRENTE – Filial – 001 – BB AGROFORT GO – 2259 | 9981-3': '14',
    'C. CORRENTE – Filial – 001 – BB AGROFORT TO – 3302 | 1155-8': '15',
    'C. CORRENTE – Filial – 001 – BB AGROFORT MS – 4490 | 6500-2': '16',
    'C. CORRENTE – Matriz – 104 – CEF CONSTRUMAX SP – 2201 | 8804-7': '17',
    'C. CORRENTE – Filial – 104 – CEF CONSTRUMAX MG – 5590 | 3044-1': '18',
    'C. CORRENTE – Filial – 104 – CEF CONSTRUMAX CE – 7822 | 9920-9': '19',
    'C. CORRENTE – Filial – 104 – CEF CONSTRUMAX PA – 4100 | 1277-3': '20',
    'C. CORRENTE – Matriz – 260 – NUBANK VISIONDATA SP – 0001 | 9901-0': '21',
    'C. CORRENTE – Filial – 260 – NUBANK VISIONDATA RJ – 0002 | 7774-2': '22',
    'C. CORRENTE – Filial – 260 – NUBANK VISIONDATA RS – 0003 | 3211-6': '23',
    'C. CORRENTE – Filial – 260 – NUBANK VISIONDATA BA – 0004 | 1199-5': '24',
    'C. CORRENTE – Matriz – 077 – INTER METALURGE MG – 0020 | 9012-1': '25',
    'C. CORRENTE – Filial – 077 – INTER METALURGE SP – 0033 | 4450-7': '26',
    'C. CORRENTE – Filial – 077 – INTER METALURGE PR – 0044 | 6608-3': '27',
    'C. CORRENTE – Filial – 077 – INTER METALURGE RS – 0055 | 2207-2': '28',
    'C. CORRENTE – Matriz – 655 – VOTORANTIM BIOENERGIA SP – 1022 | 8854-9': '29',
    'C. CORRENTE – Filial – 655 – VOTORANTIM BIOENERGIA RN – 2100 | 3310-4':'30'
}

In [4]:
# Filtro do que considerar ou não, na posição.
# Se 'Data de Recebimento2' estiver preenchida exclusivamente por data, considerar na posição
# Se 'Data de Recebimento2' estiver vazia ou Hoje ser maior do que a Data a Considerar, não considerar na posição
# Filtro novo - Considerar na Posição se:
# Se Projeção
# Data de Recebimento
# Com Valor Recebido
# 'Data a Considerar', ao inserir os dados e data de vencimento, verificar se a data de vencimento é no dia útil,
# senão, colocar em 'Data a Considerar' um data em dia útil posterior, pulando mesmo os feriados

# 'RECEBIDO', quando tiver valor na coluna 'VALOR RECEBIDO'
# 'NEGOCIAÇÃO', quando houver valor devido na coluna 'SALDO'
# 'INADIMPLENTE', quando passar da data a considerar
situacao_rcbm = ['RECEBIDO', 'A RECEBER', 'NEGOCIAÇÃO', 'CANCELADO', 'INADIMPLENTE']

clientes = ['LUCAS SÁ']  # Adicionar seguinte a lista


**Ler o Relatório Desejado**

In [5]:
# Mensagem acolhedora | Reconhecimento de Lista a ser editada  |  Identificação de área de trabalho
opcao = ['RECEITA', 'DESPESAS']
padrao_data = r'^\d{4}-\d{2}-\d{2}$'
chose = input('Bom dia, [GESTOR (A)]! \n Que você tenha um ótimo dia. \nO que vai desejar trabalhar hoje?').upper().strip()
chose = str(chose)
periodo = []

# Se ela escolher SAIR, o programa fecha imediatamente
if chose == 'CANCELAR':
    print("👋 Entendido, [GESTOR (A)]. Tenha um ótimo dia! Encerrando...")
    sys.exit()
if chose == np.nan:
    print("👋 Entendido, [GESTOR (A)]. Tenha um ótimo dia! Encerrando...")
    sys.exit()
if chose == '':
    print("👋 Entendido, [GESTOR (A)]. Tenha um ótimo dia! Encerrando...")
    sys.exit()
elif not chose in opcao:
    if not chose == '':
        while chose not in opcao:
            print(f'Opções existentes:{list(opcao)}')
            chose = input('Não existe na opcao, reescreva o que deseja').upper().strip()

print('📅 Definição do Período de Trabalho (Digite CANCELAR para fechar)')
while len(periodo) < 2:
    label = "INICIAL" if len(periodo) == 0 else "FINAL"
    dia = input(f'Digite a data {label} (AAAA-MM-DD): ').upper().strip()
    # 1. Opção de cancelar
    if dia == 'CANCELAR':
        print("🚫 Operação cancelada. Saindo...")
        sys.exit()

    # 2. Verifica se está vazio
    if not dia:
        print("⚠️ A data não pode estar vazia.")
        continue

    # 3. Valida formato e adiciona (apenas uma vez)
    # Convertemos para lower apenas no match do regex se necessário,
    # mas datas (números) não mudam com upper/lower.
    if re.match(padrao_data, dia):
        periodo.append(dia)
        print(f'✅ Data {label} registrada: {dia}')
    else:
        print(f'❌ Formato inválido! Use o padrão AAAA-MM-DD (Ex: 2021-01-15)')


# 4. Processamento dos Dados (FORA do while das datas)
print(f'\n▶️ Período selecionado: {periodo[0]} até {periodo[1]}')

# Agora sim, lemos o arquivo uma única vez
sheet1 = pd.read_excel('Relatório Financeiro 100k.xlsx', sheet_name=chose)
sheet1.columns = sheet1.columns.str.strip().str.upper()

coluna_data = 'VENCIMENTO'

sheet1[coluna_data] = pd.to_datetime(sheet1[coluna_data])

# Filtro
sheet = sheet1[
    (sheet1[coluna_data] >= periodo[0]) &
    (sheet1[coluna_data] <= periodo[1])
].copy()

if sheet.empty:
    print(f"⚠️ Nenhuma informação encontrada entre {periodo[0]} e {periodo[1]}.")
else:
    print(f"✅ {len(sheet)} linhas carregadas para o período selecionado.")
    sheet_col = sheet.columns

# Adicione isso logo após carregar o 'sheet'
sheet.columns = [str(c).strip().upper() for c in sheet.columns]
sheet_col = sheet.columns

before_insert = sheet.copy()  # Guarda o estado antes de qualquer inserção ou edição

📅 Definição do Período de Trabalho (Digite CANCELAR para fechar)
✅ Data INICIAL registrada: 2026-01-06
✅ Data FINAL registrada: 2026-01-12

▶️ Período selecionado: 2026-01-06 até 2026-01-12
✅ 105 linhas carregadas para o período selecionado.


### **2. Inserir os dados em linha específica - Manual**
* GP de pagamento não aparece PAGO se já estiver com data de pagamento

In [6]:
# Inserir dados
col_proibida = ['JUROS', 'VL. CORRIGIDO', 'TIPO DE DESPESA', 'SALDO', 'LCTO', 'SEMANA']
col_datas = ['VENCIMENTO', 'DATA A CONSIDERAR', 'EMISSAO', 'DATA DE RECEBIMENTO', 'DATA DE PAGAMENTO']
col_valores = ['SALDO', 'JUROS', 'VL. CORRIGIDO', 'VL. PARCELA', 'VL. RECEBIDO',]

exit_all = False
edit_mode = False
index_to_edit = None

selecao_col = sheet_col.difference(col_proibida, sort=False).tolist()

# Pré-processamento das contas para pegar apenas o trecho desejado
contas = {}
for nome1, id_c in bancos.items():
    partes = nome1.split(' – ')
    contas[id_c] = " – ".join(partes[2:])


row_cancelled = False
while True:
    print('Digite [a]dicionar, [e]ditar, [d]eletar [f]inalizar')
    pass_1 = input('Digite o próximo passo: ').lower().strip()
    if pass_1 == 'f':
        break
    elif pass_1 == 'd':
        confirmar = input(
            "Tem certeza que deseja apagar a ÚLTIMA linha? (S/N): ").upper(
            ).strip()
        if confirmar == 'S':
            sheet = sheet.drop(sheet.index[-1])
            print("✅ Linha removida.")
        else:
            print("❌ Operação cancelada.")
    elif pass_1 == 'a':
        edit_mode = False
        current_row_data = {col: np.nan for col in sheet_col}
    elif pass_1 == 'e':
        if sheet.empty:
            print('Não há linhas para editar, adicione.')
            continue
        ind = input('Digite o número da linha que deseja alterar').strip()
        ind = int(ind)
        if ind in sheet.index:
            edit_mode = True
            current_row_data = sheet.loc[ind].to_dict()
        else:
            print('Não existe o índice na planilha!')
            continue
    elif pass_1 == 'u':
        if sheet.empty:
            print('Não há linhas para editar, adicione.')
            continue
        edit_mode = True
        index_to_edit = len(sheet) - 1
        current_row_data = sheet.iloc[index_to_edit].to_dict()
    else:
        print('Opção Inválida.')

    row_cancelled = False
    while True:
        print(
            'Digite Concluir, Terminei, Cancelar ou a coluna que deseja editar'
        )
        print(selecao_col)
        pass_1 = input('Qual é o próximo passo: ').upper().strip()
        if pass_1 == 'TERMINEI':
            exit_all = True
        elif pass_1 == 'CANCELAR':
            row_cancelled = True
            break
        elif pass_1 == 'CONCLUIR':
            break
        elif pass_1 in col_proibida:
            print('ERRO: esta coluna é proibida para edição.')
            continue
        elif pass_1 not in sheet_col:
            print('ERRO: esta coluna não existe.')
            continue

        while True:
            if pass_1 == 'BANCO':
                print('\n--- SELECIONE UMA CONTA ---')
                for id, conta in contas.items():
                    print(f'{id}: {conta}')
                id_conta = input('Escolha um ID de conta desejada: ').strip()
                if id_conta in contas:
                    current_row_data[pass_1] = contas[id_conta]
                    break
                else:
                    print('ID Inválido!')
                    continue
            elif pass_1 == 'SITUAÇÃO' or pass_1 == 'SITUACAO':
                if chose == opcao[0]:
                    print('\n --- SELECIONE UMA SITUACAO ---')
                    for item in situacao_rcbm:
                        print(f'{item}')
                    sit = input('Escreva uma situação: ').strip().upper()
                    if sit in situacao_rcbm:
                        current_row_data[pass_1] = sit
                    elif pass_1 == 'CANCELAR':
                        print("🚫 Alteração de situação cancelada.")
                        break
                    else:
                        print('Não existe este tipo de situação')
                elif chose == opcao[1]:
                    print('\n --- SELECIONE UMA SITUACAO ---')
                    for item in situacao_pgto:
                        print(f'{item}')
                    sit1 = input('Escreva uma situação: ').strip().upper()
                    if sit1 in situacao_pgto:
                        current_row_data[pass_1] = sit1
                        break
                    else:
                        print('Não existe este tipo de situação')
                        continue

            elif pass_1 == 'FILIAL':
                print('\n --- SELECIONE UMA FILIAL ---')
                for sub in filiais:
                    print(f'{sub}')
                empresa = input('DIGITE A FILIAL: ').strip()
                # Tente converter para bater com o tipo da sua lista original
                try:
                    filial = int(empresa)
                    if filial in filiais:
                      current_row_data[pass_1] = filial
                      break
                    else:
                      print('❌ ERRO: Entrada inválida!\n Digite apenas um dos números que aparecem na lista.')
                except ValueError:
                    # Se for texto (Ex: 'Matriz')
                    print('Não é possível inserir algo além de número!')

            elif pass_1 == 'DOCUMENTO':
                doc = input('Escreva o documento (apenas números):').strip()
                try:
                    current_row_data[pass_1] = int(doc)
                    break
                except ValueError:
                    print("Erro: Digite apenas números puros, sem letras, pontos ou espaços.")
            elif pass_1 == 'FORNECEDOR':
                print('\n --- SELECIONE O FORNECEDOR ---')
                for suplyer in fornecedores:
                    print(f'{suplyer}')
                spl = input('Digite o fornecedor: ').strip()
                if spl == 'CANCELAR':
                    print("🚫 Alteração de fornecedor cancelada.")
                    break
                if spl in fornecedores:
                    current_row_data[pass_1] = spl
                    break
                else:
                    print('!!! O FORNECEDOR NÃO EXISTE !!!')
                    continue
            elif pass_1 == 'ENTIDADE':
                print('\n --- SELECIONE O CLIENTE ---')
                for customer in clientes:
                    print(f'{customer}')
                cliente = input('Digite a entidade: ').strip().upper()
                # Opção para ela desistir de alterar sem travar o programa
                if cliente == 'CANCELAR':
                    print("🚫 Alteração de data cancelada.")
                    break
                if cliente in clientes:
                    current_row_data[pass_1] = cliente
                    break
                else:
                    print('!!! A ENTIDADE NÃO EXISTE !!!')
                    continue
            elif pass_1 == 'FORMA DE RCBTO':
                print('\n --- SELECIONE A FORMA DE RECBIMENTO ---')
                for rcbm in tipos_pagamentos:
                    print(f'{rcbm}')
                rcbm_q = input('Digite a forma de Recebimento: ').strip()
                if rcbm_q in tipos_pagamentos:
                    current_row_data[pass_1] = tipos_pagamentos[rcbm_q]
                    break
                else:
                    print('!!! A FORMA ESCOLHIDA NÃO EXISTE !!!')
                    continue
            elif pass_1 == 'VENCIMENTO' or pass_1 == 'EMISSAO' or pass_1 == 'DATA DE RECEBIMENTO' or pass_1 == 'DATA DE PAGAMENTO' or pass_1 == 'DATA A CONSIDERAR':
                deadline = input('Digite a data no modelo AAAA-MM-DD: ').strip()
                if re.match(padrao_data, deadline):
                    current_row_data[pass_1] = deadline
                    print(f"✅ Data registrada: {deadline}")
                    break # Só sai se estiver certo
                else:
                    print('Formato inválido! Use AAAA-MM-DD.')
                    continue

            elif pass_1 == 'GRUPO DE PGTO'  or pass_1 == 'GRUPO DE PAGAMENTO':
                print('\n --- SELECIONE O GRUPO DE PAGAMENTO ---')
                for gp in grupos_pgto:
                    print(f'{gp}')
                gp_q = input('Digite o grupo de pagamento: ').strip()
                if gp_q in grupos_pgto:
                    current_row_data[pass_1] = gp_q
                    break
                elif pass_1 == 'CANCELAR':
                    print("🚫 Alteração de grupo de pagamento cancelada.")
                    break
                else:
                    print('!!! O GRUPO DE PAGAMENTO NÃO EXISTE !!!')
                    continue

            elif pass_1 == 'TIPO DE PAGAMENTO' or pass_1 == 'FORMA DE PAGAMENTO':
                print('\n --- SELECIONE A FORMA DE PAGAMENTO ---')
                for pgto in tipos_pagamentos:
                    print(f'{pgto}')
                pgto_q = input('Digite a forma de Pagamento: ').strip()
                if pgto_q in tipos_pagamentos:
                    current_row_data[pass_1] = pgto_q
                    break
                else:
                    print('!!! A FORMA ESCOLHIDA NÃO EXISTE !!!')
                    continue
            elif pass_1 == 'DATA A CONSIDERAR':
                if chose == 'RECEITA':
                    alvo = current_row_data.get('ENTIDADE', 'Cliente não identificado')
                else:
                    alvo = current_row_data.get('FORNECEDOR', 'Fornecedor não identificado')

                # Iniciamos um loop infinito que só será quebrado (break) com data correta
                while True:
                    data_c = input(f"Digite a nova data (DD-MM-AAAA) para o {alvo}: ").strip()

                    # Opção para ela desistir de alterar sem travar o programa
                    if data_c == 'CANCELAR':
                        print("🚫 Alteração de data cancelada.")
                        break

                    if re.match(padrao_data, data_c):
                        current_row_data[pass_1] = data_c
                        print(f"✅ Data registrada com sucesso: {data_c}")
                        break  # Sai do loop da data e volta para o menu principal
                    else:
                        print("❌ Formato inválido! Use AAAA-MM-DD.")
            else:
                # Caso a coluna seja de preenchimento livre (Ex: Histórico)
                valor_livre = input(f'Digite o conteúdo para {pass_1}: ').strip()
                current_row_data[pass_1] = valor_livre
            break
    if not row_cancelled:
        if edit_mode:
            # Substitui a linha original pelo que foi editado no dicionário
            sheet.loc[ind] = current_row_data
            print(f"✅ Linha {ind} atualizada com sucesso no sistema!")
        else:
            # Adiciona como uma nova linha se não estiver em modo de edição
            new_row = pd.DataFrame([current_row_data])
            sheet = pd.concat([sheet, new_row], ignore_index=True)
            print("✅ Nova linha adicionada com sucesso no sistema!")

# Se a [GESTOR (A)] digitar 'TERMINEI', precisamos quebrar o loop principal também
    if exit_all:
        break


Digite [a]dicionar, [e]ditar, [d]eletar [f]inalizar
Digite Concluir, Terminei, Cancelar ou a coluna que deseja editar
['VENCIMENTO', 'DOCUMENTOS', 'EMISSÃO', 'FORNECEDOR', 'REFERENTE', 'VALOR PARCELA', 'VALOR CORRIGIDO', 'DATA DE PAGAMENTO', 'GRUPO DE PAGAMENTO', 'BANCO', 'TIPO DE PAGAMENTO', 'SITUACAO', 'OBS.']
✅ Data registrada: 2026-02-13
Digite Concluir, Terminei, Cancelar ou a coluna que deseja editar
['VENCIMENTO', 'DOCUMENTOS', 'EMISSÃO', 'FORNECEDOR', 'REFERENTE', 'VALOR PARCELA', 'VALOR CORRIGIDO', 'DATA DE PAGAMENTO', 'GRUPO DE PAGAMENTO', 'BANCO', 'TIPO DE PAGAMENTO', 'SITUACAO', 'OBS.']
Digite Concluir, Terminei, Cancelar ou a coluna que deseja editar
['VENCIMENTO', 'DOCUMENTOS', 'EMISSÃO', 'FORNECEDOR', 'REFERENTE', 'VALOR PARCELA', 'VALOR CORRIGIDO', 'DATA DE PAGAMENTO', 'GRUPO DE PAGAMENTO', 'BANCO', 'TIPO DE PAGAMENTO', 'SITUACAO', 'OBS.']
Digite Concluir, Terminei, Cancelar ou a coluna que deseja editar
['VENCIMENTO', 'DOCUMENTOS', 'EMISSÃO', 'FORNECEDOR', 'REFERENTE',

### **3. Inserir dados nos DFCs - Manual**

In [7]:
import pandas as pd

file_path = 'Relatório Financeiro 100k.xlsx'

try:
    # Carregamos a estrutura do arquivo uma única vez fora do loop para performance
    excel = pd.ExcelFile(file_path)
    
    while True:
        dfc_input = input('DIGITE O DFC (AAAA-MM-DD) OU "CANCELAR": ').strip().upper()

        if dfc_input == 'CANCELAR':
            print("🚫 Operação cancelada. Saindo...")
            break

        if dfc_input in excel.sheet_names:
            print(f"✅ Sucesso! A planilha '{dfc_input}' foi encontrada.")
            df = excel.parse(dfc_input)
            break # Sai do loop após o sucesso, se desejar
        else:
            print(f"❌ Erro: A planilha '{dfc_input}' não existe.")
            print(f"💡 Abas disponíveis: {', '.join(excel.sheet_names)}")
            print("-" * 30)

except FileNotFoundError:
    print(f"🚨 Erro crítico: O arquivo '{file_path}' não foi encontrado no diretório.")

✅ Sucesso! A planilha '2026-01-06' foi encontrada.


In [34]:
# Escolher o que deseja
escolha = ['CONTA GARANTIDA - LIMITE', 'TRANSFERÊNCIA']

try:
            # --- ETAPA 1: Limpeza Inicial (Fora do Loop para Performance) ---
        for banco in lista_bancos:
            df[banco] = df[banco].astype(str)
            df[banco] = (
                df[banco]
                .str.replace('R$', '', regex=False)
                .str.replace('.', '', regex=False)
                .str.replace(',', '.', regex=False)
                .str.strip()
            )
            df[banco] = pd.to_numeric(df[banco], errors='coerce').fillna(0.0)
except Exception as e:
    print(f'ERRO: {e}')

while True:
    try:
        print(f'\nOpções existentes: {escolha}')
        entrada = input('Escolha "0" para CONTA GARANTIDA, "1" para TRANSFERÊNCIA ou digite "CANCELAR": ').strip().upper()

        # 1. Verificação de Cancelamento
        if entrada == 'CANCELAR':
            print("👋 Entendido, [GESTOR (A)]. Tenha um ótimo dia! Encerrando...")
            break 

        # 2. Conversão e Validação da Escolha
        idx = int(entrada)
        alt = escolha[idx]
        print(f'✅ Opção escolhida: {alt}')

        # Lógica dos bancos
        colunas_ignoradas = ['ATIVIDADES']
        lista_bancos = [col for col in df.columns if col not in colunas_ignoradas]
        menu_bancos = {i + 1: nome_banco for i, nome_banco in enumerate(lista_bancos)}
        
        # 3. SAÍDA DO LOOP: Se chegou aqui sem erro, podemos parar
        break

    except ValueError:
        print("❌ Entrada inválida! Digite o NÚMERO da opção ou 'CANCELAR'.")
    except IndexError:
        print(f"❌ Opção fora do intervalo! Escolha 0 ou 1.")
    except Exception as e:
        print(f"🚨 Ocorreu um erro inesperado: {e}")
        break

if not dfc_input == 'CANCELAR':
    try:
        # --- ETAPA 2: Loop de Interação ---
        while True:
            # Menu de Opções Principal
            print("\n--- MENU DE REGISTROS ---")
            print("1: Atualizar Limite (Conta Garantida)")
            print("2: Registrar Transferência")
            print("3: Sair e Salvar")
        
            # OPÇÃO 1: ATUALIZAR LIMITE
            if alt == escolha[0]:
                print('\n--- SELECIONE UMA CONTA ---')
                for id_b, conta in menu_bancos.items():
                    print(f'{id_b}: {conta}')

                id_escolhido = int(input('Escolha o ID da conta: ').strip())                
                try:
                    if id_escolhido in menu_bancos:
                        nome_coluna = menu_bancos[id_escolhido]
                        valor = input(f'Digite o novo limite para {nome_coluna} (Ex: 1500.50): ').strip()
                        
                        df.loc[df['ATIVIDADES'] == 'CONTA GARANTIDA - LIMITE', nome_coluna] = valor
                        print(f"✅ Limite de {nome_coluna} atualizado!")
                    else:
                        print("❌ ID não encontrado.")
                except ValueError:
                    print("❌ Erro: Digite apenas números para ID e Valor.")

            # OPÇÃO 2: TRANSFERÊNCIA
            elif alt == escolha[1]:
                print('\n--- SELECIONE AS CONTAS ---')
                for id_b, col in menu_bancos.items():
                    print(f'{id_b}: {col}')
                
                try:
                    id_origem = int(input('ID da conta ORIGEM: ').strip())
                    id_destino = int(input('ID da conta DESTINO: ').strip())

                    conta_origem = menu_bancos.get(id_origem)
                    conta_destino = menu_bancos.get(id_destino)

                    if conta_origem and conta_destino:
                        valor_envio = float(input(f'Valor da transferência: ').strip())
                        
                        confirmar = input(f'Confirmar R$ {valor_envio:.2f} de {conta_origem} para {conta_destino}? (S/N): ').upper()
                        if confirmar == 'S':
                            df.loc[df['ATIVIDADES'] == 'TRANSFERÊNCIA ENVIADA', conta_origem] = valor_envio
                            df.loc[df['ATIVIDADES'] == 'TRANSFERÊNCIA RECEBIDA', conta_destino] = valor_envio
                            print("✅ Transferência registrada!")
                    else:
                        print("❌ Um ou ambos os IDs são inválidos.")
                except ValueError:
                    print("❌ Erro: Entrada de dados inválida.")

            # OPÇÃO 3: SAIR
            elif alt == '3':
                print("Encerrando edições...")
                break
            else:
                print("Opção inválida!")

        # Finalização fora do While
        df[lista_bancos] = df[lista_bancos].replace(0.0, np.nan).fillna('') # Devolve vazio para estética, se preferir
        print("\n✅ Processamento concluído com sucesso!")

    except Exception as e:
        print(f"🚨 Ocorreu um erro inesperado: {e}")
else:
    print("👋 Entendido, [GESTOR (A)]. Encerrando...")


Opções existentes: ['CONTA GARANTIDA - LIMITE', 'TRANSFERÊNCIA']
❌ Opção fora do intervalo! Escolha 0 ou 1.

Opções existentes: ['CONTA GARANTIDA - LIMITE', 'TRANSFERÊNCIA']
✅ Opção escolhida: TRANSFERÊNCIA

--- MENU DE REGISTROS ---
1: Atualizar Limite (Conta Garantida)
2: Registrar Transferência
3: Sair e Salvar

--- SELECIONE AS CONTAS ---
1: C. CORRENTE – Filial – 104 – CEF CONSTRUMAX MG – 5590 | 3044-1
2: C. CORRENTE – Filial – 655 – VOTORANTIM BIOENERGIA RN – 2100 | 3310-4
3: C. CORRENTE – Filial – 033 – SANTANDER TECHPLUS PR – 7720 | 6509-9
4: C. CORRENTE – Filial – 104 – CEF CONSTRUMAX PA – 4100 | 1277-3
5: C. CORRENTE – Matriz – 077 – INTER METALURGE MG – 0020 | 9012-1
6: C. CORRENTE – Matriz – 033 – SANTANDER TECHPLUS MG – 6400 | 3528-4
7: C. CORRENTE – Filial – 077 – INTER METALURGE PR – 0044 | 6608-3
8: C. CORRENTE – Filial – 237 – BRADESCO MEGASTORE BA – 5602 | 7010-6
9: C. CORRENTE – Filial – 237 – BRADESCO MEGASTORE RJ – 3344 | 2001-8
10: C. CORRENTE – Filial – 001 – BB

### **4. Negociações dos Clientes ou Fornecedores que foram inseridos:**
Escolher:
* As linhas que são para apagar;
* Os dados que deseja alterar.
???

In [ ]:
# Após a etapa de inserção, vamos identificar as linhas que estão em NEGOCIAÇÃO para destacar posteriormente
pos_sit = ['NEGOCIACAO', 'NEGOCIAÇÃO', 'INADIMPLENTE']
col_situacao1 = ['Situação', 'SITUACAO']
col_situaçãoid = 0 if chose == opcao[0] else 1
col_negociacao = sheet.columns
negociacao = pd.DataFrame(columns=col_negociacao)
original = sheet1.copy()
col_fe = ['ENTIDADE', 'FORNECEDOR']
col_fe_id = 0 if chose == opcao[0] else 1
length = len(sheet) - len(before_insert)
sheet_end = sheet.tail(length)

try:
    if not original.empty:
        if original[col_situacao1[col_situaçãoid]].isin(pos_sit).any():
            if not sheet_end.empty:
                if length > 0:
                    for row in original.index:
                        if original.loc[row, col_situacao1[col_situaçãoid]] in pos_sit and original.loc[row, col_fe[col_fe_id]] in sheet_end[col_fe[col_fe_id]].values:
                            negociacao = pd.concat([negociacao, original.loc[row].to_frame().T], ignore_index=True)
                    print("\n✅ Identificação de negociações concluída.")
    else:    
        print("⚠️ A planilha original está vazia, não foi possível identificar negociações anteriores.")

except KeyError as e:
    print(f"🚨 Erro de chave: A coluna {e} não foi encontrada. Verifique os nomes das colunas na planilha.")
except Exception as e:
    print(f"🚨 Ocorreu um erro inesperado: {e}")

negociacao

In [ ]:
if not negociacao.empty and negociacao is not None:
    while True:
        print("\nO que deseja fazer com as negociações encontradas?")
        print("1: Exportar para Excel")
        print("2: Ignorar por enquanto")
        decisao = input("Digite o número da opção desejada: ").strip()
        
        if decisao == '1':
            nome_arquivo = f"negociacoes_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
            negociacao.to_excel(nome_arquivo, index=False)
            print(f"✅ Negociações exportadas para {nome_arquivo}")
            break
        elif decisao == '2':
            print("👋 Entendido, [GESTOR (A)]. Tenha um ótimo dia! Encerrando...")
            break
        else:
            print("Opção inválida. Por favor, escolha 1 ou 2.")

In [ ]:
for row in negociacao.index:
    soma = negociacao.loc[row, col_fe.loc[col_fe_id] in sheet_end.loc[col_fe[col_fe_id]]].sum
    print(f"Soma para a linha {row}: {soma}")

### **5. Formatar dados**

In [ ]:
#################     Formatar dados     #################
# --- Funções Auxiliares (Mantidas) ---
def clean_string(s):
    if not isinstance(s, str):
        return str(s)
    normalized_s = unicodedata.normalize('NFKD', s)
    cleaned_s = normalized_s.encode('ascii', 'ignore').decode('utf-8')
    cleaned_s = re.sub(r'\s+', ' ', cleaned_s).strip()
    return cleaned_s

def safe_loc(df, label, selector=None):
    if label and label in df.index:
        selected = df.loc[label].fillna(0)
        if isinstance(selected, pd.DataFrame):
            if selector == 'first': return selected.iloc[0]
            elif selector == 'last': return selected.iloc[-1]
            else: return selected.sum()
        return selected
    return pd.Series(0.0, index=df.columns)


def get_previous_day_specific_value(current_date, excel_file_path, bank_columns_names, target_activity_label):
    previous_date = current_date - timedelta(days=1)
    while previous_date.weekday() >= 5: # 5=Sábado, 6=Domingo
        previous_date -= timedelta(days=1)
    
    previous_sheet_name = previous_date.strftime('%Y-%m-%d')
    saldo_values = pd.Series(0.0, index=bank_columns_names)

    try:
        xls = pd.ExcelFile(excel_file_path)
        if previous_sheet_name in xls.sheet_names:
            prev_day_dfc = pd.read_excel(excel_file_path, sheet_name=previous_sheet_name)
            if 'ATIVIDADES' in prev_day_dfc.columns:
                prev_day_dfc['ATIVIDADES'] = prev_day_dfc['ATIVIDADES'].apply(clean_string)
                target_row = prev_day_dfc[prev_day_dfc['ATIVIDADES'] == target_activity_label]
                if not target_row.empty:
                    common_cols = list(set(bank_columns_names) & set(target_row.columns))
                    saldo_values[common_cols] = pd.to_numeric(target_row[common_cols].iloc[0], errors='coerce').fillna(0.0)
    except Exception as e:
        print(f"Erro ao buscar valor do dia anterior ({target_activity_label}): {e}")
    return saldo_values

# --------------------------------------------------
# Função de extração de conta
# --------------------------------------------------
def extrair_conta(texto):
    padrao = r"(\d{3,5}\s*\|?\s*\d{3,6}-\d)"
    match = re.findall(padrao, str(texto))
    if not match:
        return None
    return match[-1].replace(" ", "").replace("|", "")





# Converter colunas de datas em datas
try:
    print(f'INCIANDO A FORMATAÇÃO DE DADOS DA PLANILHA {chose}!\nPor Favor, Certifique do arquivo estar fechado e não interrompa o processo!')

    if not sheet.equals(sheet1):
        for nome_esperado in col_datas:
            # Verifica se o nome esperado existe nas colunas da planilha buscada
            if nome_esperado in sheet_col:
                print(f"Convertendo a coluna: {nome_esperado}")
                # Transforma a coluna da planilha 'sheet' em formato de data real
                sheet[nome_esperado] = pd.to_datetime(sheet[nome_esperado], errors='coerce')
        # Converter colunas de valores em valores
        try:
            for valor in col_valores:
                # Verifica se o nome esperado existe nas colunas da planilha buscada
                if valor in sheet_col:
                    sheet[valor] = pd.to_numeric(sheet[valor], errors='coerce')
                    print(f"Convertendo a coluna: {valor}")
            sheet['VALOR CORRIGIDO'] = sheet['VALOR DA PARCELA']
            sheet['VALOR CORRIGIDO'] = sheet['JUROS'].fillna(sheet['VALOR CORRIGIDO'])
            ambos_preenchidos = sheet['JUROS'].notnull() & sheet['VALOR DA PARCELA'].notnull()
            sheet.loc[ambos_preenchidos, 'VALOR CORRIGIDO'] = sheet['JUROS'] - sheet['VALOR DA PARCELA']
            print(f"✅ Coluna VALOR CORRIGIDO calculada com sucesso linha por linha.")
        except KeyError: pass

    if chose == opcao[1]:
        try:
            sheet['TIPO DE DESPESA'] = ''
            for despesa in reconhecimento_despesas:
                # Transformando seu formato atual no formato que o Pandas entende
                mapeamento_final = {}
                for categoria, itens in reconhecimento_despesas.items():
                    for item in itens:
                        mapeamento_final[item] = categoria

                # Aplica o reconhecimento automático na coluna TIPO DE DESPESA
                sheet['TIPO DE DESPESA'] = sheet['GRUPO DE PAGAMENTO'].map(mapeamento_final).fillna('OUTRAS DESPESAS')
                # Remova o 'break' se quiser que ele classifique TODAS as categorias do dicionário

            # Preenche a coluna CONTA_LIMPA, se não existir, cria e coloca.
            sheet["CONTA_LIMPA"] = sheet["BANCO"].apply(extrair_conta)

        except KeyError: # Use KeyError se a coluna não existir, ou Exception para qualquer erro
            print('A coluna GRUPO DE PAGAMENTO não foi encontrada!')
    elif chose == opcao[0]:
        try:
            # 1. Lista de termos que definem que NÃO é um cliente comum
            termos_operacionais = ['RESGATE', 'RENDIMENTO', 'DEVOLUÇÃO', 'ESTORNO']

            # 2. Função que decide o que escrever na coluna
            def definir_tipo(entidade):
                entidade_upper = str(entidade).upper()
                # Verifica se algum termo da lista está dentro do nome da entidade
                for termo in termos_operacionais:
                    if termo in entidade_upper:
                        return termo  # Se achou (ex: RESGATE), retorna o próprio termo
                
                return 'CLIENTE'  # Se percorreu a lista e não achou nada, é CLIENTE

            # 3. Criar a coluna de uma vez só
            sheet['TIPO DE RECEITA'] = sheet['ENTIDADE'].apply(definir_tipo)
            for receita in reconhecimento_receitas:
                # Transformando seu formato atual no formato que o Pandas entende
                mapeamento_final = {}
                for categoria, itens in reconhecimento_receitas.items():
                    for item in itens:
                        mapeamento_final[item] = categoria
        except KeyError: # Use KeyError se a coluna não existir, ou Exception para qualquer erro
            print('A coluna ENTIDADE não foi encontrada!')
    sheet.tail()
except NameError:
    print('Erro: Chose não foi definido!')



try:
    print(f'INCIANDO A FORMATAÇÃO DE DADOS DO {DFC}!\nPor Favor, Certifique do arquivo estar fechado e não interrompa o processo!')

    # --- 1. Padronização e Identificação de Colunas ---
    if df.index.name == 'ATIVIDADES':
        df.reset_index(inplace=True)

    df['ATIVIDADES'] = df['ATIVIDADES'].apply(clean_string)

    # Define colunas de bancos (dinâmico) e garante que SALDOS BANCOS exista
    bank_columns = [col for col in df.columns if col not in ['ATIVIDADES', 'SALDOS BANCOS']]
    if 'SALDOS BANCOS' not in df.columns:
        df['SALDOS BANCOS'] = 0.0

    # --- 2. Limpeza Numérica Única ---
    for col in bank_columns:
        df[col] = df[col].astype(str).str.replace('R$', '', regex=False)\
                        .str.replace('.', '', regex=False)\
                        .str.replace(',', '.', regex=False).str.strip()
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

    current_dfc_date = pd.to_datetime(DFC)

    # --- 3. Cálculos Iniciais e Dia Anterior ---
    current_bank_columns = bank_columns
    saldo_xpto_values = get_previous_day_specific_value(current_dfc_date, 'Relatório Financeiro 100k.xlsx', current_bank_columns, clean_string('DISPONIBILIDADES C/C'))
    previous_day_total_aplicado_values = get_previous_day_specific_value(current_dfc_date, 'Relatório Financeiro 100k.xlsx', current_bank_columns, clean_string('TOTAL APLICADO'))

    despesa = pd.read_excel('Relatório Financeiro 100k.xlsx', sheet_name= 'DESPESA')
    # --- 4. Loop de Grupos de Pagamento (Modelo Original) ---


    # --------------------------------------------------
    # 1. Mapa de contas → colunas do DFC
    # --------------------------------------------------
    mapa_colunas = {
        extrair_conta(col): col
        for col in df.columns
        if col != "ATIVIDADES" and extrair_conta(col) is not None
    } 

    all_payment_groups = despesa['GRUPO DE PAGAMENTOS'].unique()

    despesa_filtered_by_date = despesa[despesa["VENCIMENTO"] == current_dfc_date].copy()
    
    for idx, atividade in df["ATIVIDADES"].items():
        if atividade in all_payment_groups:
            despesa_atividade = despesa_filtered_by_date[despesa_filtered_by_date["GRUPO DE PAGAMENTO"] == atividade]
            for conta_limpa, coluna_DFC in mapa_colunas.items():
                if coluna_DFC in df.columns:
                    soma = despesa_atividade.loc[despesa_atividade["CONTA_LIMPA"] == conta_limpa, "VALOR CORRIGIDO"].sum()
                    df.loc[idx, coluna_DFC] = soma

    # --- 5. Cálculos de Linhas de Saldo (Setando Index para safe_loc) ---
    df.set_index('ATIVIDADES', inplace=True)

    # Definição de Labels
    labels = {
        'xpto': clean_string('SALDO XPTO'),
        'entradas_ant': clean_string('ENTRADAS DIA ANTERIOR (+)'),
        'saldos_bancos': clean_string('SALDOS BANCOS'),
        'bloqueado': clean_string('SALDO BLOQUEADO'),
        'empresa': clean_string('SALDOS EMPRESA'),
        'saidas_dia': clean_string('SAÍDAS DO DIA'),
        'deposito_conta': clean_string('SALDO DEPOSITO EM CONTA'),
        'disp_cc': clean_string('DISPONIBILIDADES C/C'),
        'disp_geral': clean_string('DISPONIBILIDADES GERAL'),
        'total_aplic': clean_string('TOTAL APLICADO'),
        'resgate': clean_string('Resgate'),
        'aplic_inv': clean_string('Aplicação')
    }

    # SALDO XPTO e ENTRADAS
    if labels['xpto'] in df.index:
        df.loc[df.index == labels['xpto'], current_bank_columns] = saldo_xpto_values.values
        if labels['entradas_ant'] in df.index:
            df.loc[df.index == labels['entradas_ant'], current_bank_columns] = safe_loc(df, labels['xpto']).values

    # SALDO DEPOSITO EM CONTA
    if labels['deposito_conta'] in df.index:
        v_rec = safe_loc(df, clean_string('TRANSFERÊNCIA RECEBIDA'))
        v_env = safe_loc(df, clean_string('TRANSFERÊNCIA ENVIADA'))
        df.loc[df.index == labels['deposito_conta'], current_bank_columns] = (v_rec - v_env).values

    # DISPONIBILIDADE C/C
    if labels['disp_cc'] in df.index:
        df.loc[df.index == labels['disp_cc'], current_bank_columns] = (
            safe_loc(df, labels['empresa']) - 
            safe_loc(df, labels['saidas_dia']) + 
            safe_loc(df, labels['deposito_conta'])
        ).values

    # --- 6. TRATAMENTO DAS 3 DISPONIBILIDADES GERAL ---
    mask_disp_geral = np.where(df.index.astype(str).str.strip() == labels['disp_geral'])[0]

    if len(mask_disp_geral) > 0:
        # 1ª Ocorrência (Topo): DISPONIBILIDADES C/C + TOTAL APLICADO
        val_topo = safe_loc(df, labels['disp_cc']) + safe_loc(df, labels['total_aplic'])
        df.iloc[mask_disp_geral[0], df.columns.get_indexer(current_bank_columns)] = val_topo.values

        # Ocorrência do Meio (Penúltima): Busca do Dia Anterior
        if len(mask_disp_geral) >= 2:
            val_meio = get_previous_day_specific_value(
                current_date=current_dfc_date,
                excel_file_path="Relatório Financeiro 100k.xlsx",
                bank_columns_names=current_bank_columns,
                target_activity_label=labels['disp_geral']
            )
            df.iloc[mask_disp_geral[-2], df.columns.get_indexer(current_bank_columns)] = val_meio.values

        # Última Ocorrência (Fundo): Resgate - Aplicações
        val_fundo = (
            safe_loc(df, labels['disp_geral'])[1] # Referência ao valor anterior conforme seu modelo
            - safe_loc(df, labels['resgate'], selector='last')
            + safe_loc(df, labels['aplic_inv'], selector='last')
        )
        df.iloc[mask_disp_geral[-1], df.columns.get_indexer(current_bank_columns)] = val_fundo.values

    # --- 7. Finalização e Soma Horizontal ---
    df.reset_index(inplace=True)
    df['SALDOS BANCOS'] = df[bank_columns].sum(axis=1)

    print('DFC formatado com sucesso!')
    df
except NameError:
    print('ERRO: O DFC não foi definido!')

col_situacao1 = ['Situação', 'SITUACAO']
col_situaçãoid = 0 if chose == opcao[0] else 1

try:
    if not sheet.empty and chose == opcao[1]:
        # Cria uma máscara onde a data de vencimento é menor que a data de pagamento
        atrasados = sheet['VENCIMENTO'] < sheet['DATA DE PAGAMENTO']
        # Atualiza a coluna 'SITUAÇÃO' apenas nessas linhas
        sheet.loc[atrasados, col_situacao1[col_situaçãoid]] = 'PAGO ATRASADO'
    elif not sheet.empty and chose == opcao[0]:
        # Cria uma máscara onde a data de vencimento é menor que a data de pagamento
        atrasados = sheet['VENCIMENTO'] < sheet['DATA DE RBCTO']
        # Atualiza a coluna 'SITUAÇÃO' apenas nessas linhas
        sheet.loc[atrasados, col_situacao1[col_situaçãoid]] = 'RCEBIDO ATRASADO'
except Exception as e:
    print(e)


Além de formatar, caso atinja o limite de 100k linhas em Receita ou Despesa, cria um novo com os últimos 30 DFCs junto com a RECEITA e DESPESA:

In [ ]:
# FUNÇÕES: Formatação de dados
def format_currency(df, columns):
    """Formata colunas numéricas de um DataFrame para o formato monetário 'R$ X.XX'."""
    df_copy = df.copy()
    for col in columns:
        if col in df_copy.columns:
            # Converte para numérico, preenchendo NaNs com 0 para evitar erros na formatação
            df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce').fillna(0)
            df_copy[col] = df_copy[col].apply(lambda x: f'R$ {x:.2f}')
    return df_copy

def format_dates(df, columns):
    """Formata colunas de data para o formato 'YYYY-MM-DD'."""
    df_copy = df.copy()
    for col in columns:
        if col in df_copy.columns:
            df_copy[col] = pd.to_datetime(df_copy[col], errors='coerce').dt.strftime('%Y-%m-%d')
    return df_copy

try:
    # Formatação das colunas de valores e datas em RECEITA e DESPESA
    try:
        print(f'Realizando a formatação em {chose} ...')
        if chose == opcao[0]:
            # --- Preparar RECEITA ---
            receita_to_export = pd.read_excel('Relatório Financeiro 100k.xlsx', sheet_name='RECEITA').copy() # Use a cópia global RECEITA
            receita_to_export = format_currency(receita_to_export, col_valores)
            receita_to_export = format_dates(receita_to_export, col_datas)
        elif chose == opcao[1]:
            # --- Preparar DESPESA ---
            despesa_to_export = pd.read_excel('Relatório Financeiro 100k.xlsx', sheet_name='DESPESAS').copy() # Use a cópia global despesa
            despesa_to_export = format_currency(despesa_to_export, col_valores)
            despesa_to_export = format_dates(despesa_to_export, col_datas)
    except NameError:
        print('Não foi feito alteração na RECEITA ou na DESPESA!')


    # Garante que as colunas vazias no DFC (ex: 'ATIVIDADES'=='') sejam vazias e não 0.0
    # Isso é para evitar que linhas vazias sejam exportadas como 'R$ 0.00'
    dfc_formatted_for_export = df.copy()
    for col in df.columns:
        if 'ATIVIDADES' in df.columns and df[df['ATIVIDADES'] == ''].shape[0] > 0:
            dfc_formatted_for_export.loc[df['ATIVIDADES'] == '', col] = ''

        # --- Nome do arquivo Excel e data atual para nome das abas DFC futuras ---
        excel_file_name = 'Relatório Financeiro 100k.xlsx'

        # --- Exportar para Excel ---
        # Alterado engine para 'openpyxl' para suportar o modo 'a' (append) e if_sheet_exists='replace'
        with pd.ExcelWriter(excel_file_name, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            # Verifica se a aba 'DFC' deve ser atualizada para o nome da data atual
            today_sheet_name = date.today().strftime('%Y-%m-%d')
            try:    
                dfc_formatted_for_export.to_excel(writer, sheet_name=today_sheet_name, index=False)
            except NameError: print('ERRO: DFC não selecionado.')
            try:
                despesa_to_export.to_excel(writer, sheet_name='DESPESAS', index=False)
            except NameError: print('ERRO: DESPESA não selecionado.')
            try:
                receita_to_export.to_excel(writer, sheet_name='RECEITA', index=False)
            except NameError: print('ERRO: DFC não selecionado.')
        print(f"DataFrames exportados com sucesso para '{excel_file_name}' nas abas: {today_sheet_name}, RECEITA e DESPESA")
except Exception as e:
    print(f'Houve um erro ao tentar formatar, verifique {e}.')



def realizar_manutencao(caminho_arquivo):
    limite_linhas = 100000
    reserva_historico = 15000
    
    # 1. Carrega as abas principais para verificação
    xls = pd.ExcelFile(caminho_arquivo) # Usamos ExcelFile para ver todas as abas
    df_despesas = pd.read_excel(xls, sheet_name='DESPESAS') # Ajustado de 'DESPESAS' para 'DESPESA' conforme seu código anterior
    df_receita = pd.read_excel(xls, sheet_name='RECEITA')
    
    if len(df_despesas) > limite_linhas or len(df_receita) > limite_linhas:
        print("🚀 Limite atingido! Criando novo banco e arquivando antigo...")
        
        timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M')
        nome_backup = caminho_arquivo.replace(".xlsx", f"POSICAO_{timestamp}.xlsx")
        
        # Copia o arquivo atual para backup
        import shutil
        shutil.copy(caminho_arquivo, nome_backup)
        
        # --- LÓGICA PARA OS 30 DFCs ---
        # Filtra abas que tem nome de data (ex: 2024-05-20)
        todas_abas = xls.sheet_names
        abas_dfc = [aba for aba in todas_abas if aba not in ['DESPESAS', 'RECEITA']]
        # Ordenamos e pegamos as últimas 30
        abas_dfc_recentes = sorted(abas_dfc)[-31:]
        
        # 2. Cria o novo "arquivo limpo"
        with pd.ExcelWriter(caminho_arquivo, engine='openpyxl') as writer:
            # Mantém os últimos registros de Receita e Despesa
            df_despesas.tail(reserva_historico).to_excel(writer, sheet_name='DESPESAS', index=False)
            df_receita.tail(reserva_historico).to_excel(writer, sheet_name='RECEITA', index=False)
            
            # 3. Re-insere os últimos 30 DFCs
            print(f"Copiando as últimas {len(abas_dfc_recentes)} abas de DFC...")
            for aba in abas_dfc_recentes:
                df_temp = pd.read_excel(xls, sheet_name=aba)
                df_temp.to_excel(writer, sheet_name=aba, index=False)
            
            print("✅ Manutenção concluída. Histórico e últimos 30 DFCs preservados.")
    else:
        print("Everything is fine. O arquivo ainda tem espaço.")

# --- CHAMADA DA FUNÇÃO ---
meu_arquivo = "Relatório Financeiro 100k.xlsx"
realizar_manutencao(meu_arquivo)


### **6. Transferir para o excel**

In [ ]:
# --- Configuração do Log ---
def salvar_log(mensagem):
    timestamp = datetime.now().strftime('%H:%M:%S')
    with open("log_processamento.txt", "a", encoding="utf-8") as f:
        f.write(f"[{timestamp}] {mensagem}\n")

print("💾 Salvando alterações no arquivo...")

try:
    # O engine 'openpyxl' com overlay permite manter formatações prévias
    with pd.ExcelWriter('Relatório Financeiro 100k.xlsx', engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:
        
        # 1. Salvar aba de Edição (RECEITA ou DESPESA)
        try:
            if 'chose' in locals() and sheet is not None:
                sheet.to_excel(writer, sheet_name=chose, index=False)
                msg = f'✅ Aba {chose} atualizada com dados brutos.'
                print(msg)
                salvar_log(msg)
            
            # Pequena pausa para o sistema de arquivos respirar
            time.sleep(1) 
            
        except NameError:
            print('⚠️ Não foi definido o chose!')

        # 2. Salvar aba do DFC (Consolidado)
        try:
            if 'DFC' in locals() and df is not None:
                df.to_excel(writer, sheet_name=DFC, index=False)
                msg = f'✅ Aba {DFC} (DFC) atualizada com sucesso.'
                print(msg)
                salvar_log(msg)
        except NameError:
            print('⚠️ Não foi definido o DFC!')

    print("🚀 Processo de persistência concluído!")

except PermissionError:
    print("❌ ERRO CRÍTICO: O arquivo 'Relatório Financeiro 100k 100k 100k 100k 100k 100k 100k 100k 100k 100k.xlsx' está aberto. Feche-o e tente novamente.")
    salvar_log("ERRO: Arquivo aberto pelo usuário.")
except Exception as e:
    print(f"❌ Erro inesperado ao salvar: {e}")
    salvar_log(f"ERRO: {str(e)}")

Se for necessário alinhar as últimas 7 planilhas, incluindo RECEITA e DESPESA, tire as aspas triplas ''', no iníco e no final, e ative o código abaixo.

In [ ]:
'''# --- 1. FUNÇÕES AUXILIARES DE FORMATAÇÃO (O SEU "ESCUDO") ---
def format_currency(df, columns):
    """Transforma números em texto 'R$ X.XX' para travar a modelagem no Excel."""
    df_copy = df.copy()
    for col in columns:
        if col in df_copy.columns:
            # Converte para numérico antes de formatar para evitar erros
            df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')
            df_copy[col] = df_copy[col].apply(lambda x: f'R$ {x:.2f}' if pd.notna(x) else '')
    return df_copy

def format_dates(df, columns):
    """Padroniza datas para o formato AAAA-MM-DD."""
    df_copy = df.copy()
    for col in columns:
        if col in df_copy.columns:
            df_copy[col] = pd.to_datetime(df_copy[col], errors='coerce').dt.strftime('%Y-%m-%d')
    return df_copy

# --- 2. PREPARAÇÃO DOS DADOS TIPO 'RECEITA' E 'DESPESA' ---
# Nota: Estou assumindo que os DataFrames RECEITA e despesa já estão carregados no seu ambiente.
receita_to_export = format_currency(RECEITA.copy(), ['Valor da parcela', 'Valor Recebido', 'Valor Corrigido', 'Saldo', 'JUROS'])
receita_to_export = format_dates(receita_to_export, ['Vencimento', 'Emissão', 'Data a considerar'])

despesa_to_export = format_currency(despesa.copy(), ['VALOR PARCELA', 'JUROS', 'VALOR CORRIGIDO'])
despesa_to_export = format_dates(despesa_to_export, ['VENCIMENTO', 'EMISSÃO', 'DATA DE PAGAMENTO'])

# --- 3. GESTÃO INTELIGENTE DE ABAS (FOCO NAS ÚLTIMAS 7) ---
excel_file_name = 'Relatório Financeiro 100k.xlsx'

try:
    # Lê apenas os nomes das abas para não sobrecarregar a memória
    xls = pd.ExcelFile(excel_file_name)
    todas_as_abas = xls.sheet_names

    # Filtra abas que seguem o padrão de data (DFC) e ordena cronologicamente
    abas_data = sorted([a for a in todas_as_abas if re.match(r'\d{4}-\d{2}-\d{2}', a)])

    # Seleciona apenas as últimas 7 abas de DFC para processamento
    ultimas_7_abas = abas_data[-7:] if len(abas_data) >= 7 else abas_data

    # Carrega o conteúdo apenas dessas 7 abas selecionadas
    dfc_sheets_to_process = []
    for nome_aba in ultimas_7_abas:
        df_lido = pd.read_excel(excel_file_name, sheet_name=nome_aba)
        dfc_sheets_to_process.append((nome_aba, df_lido))

    # --- 4. EXPORTAÇÃO VIA OVERLAY (PRECISÃO CIRÚRGICA) ---
    # mode='a' e overlay permitem atualizar abas sem deletar as outras 750+ abas existentes
    with pd.ExcelWriter(excel_file_name, engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:
        # Carrega o "esqueleto" do arquivo atual
        writer.book = openpyxl.load_workbook(excel_file_name)
        writer.sheets = {ws.title: ws for ws in writer.book.worksheets}

        # Atualiza e formata como texto apenas os últimos 7 DFCs
        for nome_aba, df_conteudo in dfc_sheets_to_process:
            # Identifica colunas de valor (tudo exceto a descrição da atividade)
            colunas_num = [c for c in df_conteudo.columns if c != 'ATIVIDADES']
            df_formatado = format_currency(df_conteudo, colunas_num)

            # Grava na aba correspondente
            df_formatado.to_excel(writer, sheet_name=nome_aba, index=False)

        # Atualiza as abas fixas de controle
        receita_to_export.to_excel(writer, sheet_name='RECEITA', index=False)
        despesa_to_export.to_excel(writer, sheet_name='DESPESA', index=False)

    # --- 5. DEFINIR A ABA ATIVA (FOCO NO TRABALHO DA [GESTOR (A)]) ---
    # Reabrimos para ajustar o ponteiro da aba ativa
    wb = openpyxl.load_workbook(excel_file_name)
    if 'DESPESA' in wb.sheetnames:
        # Encontra o índice da aba DESPESA e a torna ativa
        idx = wb.sheetnames.index('DESPESA')
        wb.active = idx
        wb.save(excel_file_name)
        print(f"✨ Sucesso! {len(ultimas_7_abas)} abas de DFC atualizadas.")
        print(f"📁 Relatório pronto com 'DESPESA' como aba principal.")
    else:
        print("⚠️ Aviso: Abas atualizadas, mas aba 'DESPESA' não foi encontrada para ativação.")

except FileNotFoundError:
    print(f"❌ Erro: O arquivo '{excel_file_name}' não existe. Certifique-se do nome do arquivo.")
except Exception as e:
    print(f"⚠️ Ocorreu um erro inesperado: {e}")

''';

### **7. Criar DFCs até 3, sem contar fins de semana e feriados.**

In [ ]:
# --- 1. FUNÇÕES DE APOIO ---
def format_currency(df, columns):
    df_copy = df.copy()
    for col in columns:
        if col in df_copy.columns:
            df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce').fillna(0)
    return df_copy

def generate_and_populate_dfc_for_date(target_date, base_dfc_activities, despesa_df, receita_df):
    bancos_ativos = sorted(list(set(receita_df['Banco'].unique()) | set(despesa_df['BANCO'].unique())))
    bancos_ativos = [b for b in bancos_ativos if str(b) != 'nan' and b != '']
    
    df_dia = pd.DataFrame({'ATIVIDADES': base_dfc_activities})
    for banco in bancos_ativos:
        df_dia[banco] = 0.0
    
    target_str = target_date.strftime('%Y-%m-%d')

    # Lançar Receitas - Ajustado para 'VENCIMENTO'
    if 'VENCIMENTO' in receita_df.columns:
        rec_hoje = receita_df[pd.to_datetime(receita_df['VENCIMENTO']).dt.strftime('%Y-%m-%d') == target_str]
        for _, linha in rec_hoje.iterrows():
            cat, banco, valor = linha['TIPO DE RECEITA'], linha['Banco'], linha['VALOR RECEBIDO']
            if cat in df_dia['ATIVIDADES'].values and banco in df_dia.columns:
                df_dia.loc[df_dia['ATIVIDADES'] == cat, banco] += valor

    # Lançar Despesas - Ajustado para 'DATA DE PAGAMENTO'
    if 'DATA DE PAGAMENTO' in despesa_df.columns:
        desp_hoje = despesa_df[pd.to_datetime(despesa_df['DATA DE PAGAMENTO']).dt.strftime('%Y-%m-%d') == target_str]
        for _, linha in desp_hoje.iterrows():
            cat, banco, valor = linha['GRUPO DE PAGAMENTO'], linha['BANCO'], linha['VALOR PAGO']
            if cat in df_dia['ATIVIDADES'].values and banco in df_dia.columns:
                df_dia.loc[df_dia['ATIVIDADES'] == cat, banco] += valor

    df_dia['SALDO BANCOS'] = df_dia[bancos_ativos].sum(axis=1)
    
            
    return df_dia

# --- 1. FUNÇÕES DE APOIO ---
def clean_numeric_columns(df, columns):
    df_copy = df.copy()
    for col in columns:
        if col in df_copy.columns:
            # Converte para numérico, forçando erros para NaN
            df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce').fillna(0)
    return df_copy

# --- 2. LEITURA E HIGIENIZAÇÃO ---
excel_file_name = 'Relatório Financeiro 100k.xlsx'

# Tenta ler as abas; se houver BadZipFile, feche o Excel e tente novamente
despesa = pd.read_excel(excel_file_name, sheet_name='DESPESAS')
RECEITA = pd.read_excel(excel_file_name, sheet_name='RECEITA')

# Classificação de Receita
termos_operacionais = ['RESGATE', 'RENDIMENTO', 'DEVOLUÇÃO', 'ESTORNO']
def definir_tipo(entidade):
    entidade_upper = str(entidade).upper()
    for termo in termos_operacionais:
        if termo in entidade_upper: return termo
    return 'CLIENTE'

RECEITA['TIPO DE RECEITA'] = RECEITA['Entidade'].apply(definir_tipo)

sit_block = ['NEGOCIAÇÃO', 'VENCIDO', 'INADIMPLENTE', 'CANCELADO']
RECEITA = RECEITA[(~RECEITA['Situação'].isin(sit_block)) & (RECEITA['Valor Recebido'].notna())].copy()
despesa = despesa[(~despesa['SITUACAO'].isin(sit_block)) & (despesa['DATA DE PAGAMENTO'].notna())].copy()

# --- 3. TEMPLATE E DATAS ---
initial_dfc_activities_list = []
with pd.ExcelFile(excel_file_name) as xls:
    abas_dfc = [s for s in xls.sheet_names if s not in ['BANCO DE DADOS', 'DESPESAS', 'RECEITA']]
    if abas_dfc:
        ultimo_dfc = pd.read_excel(xls, sheet_name=abas_dfc[-1])
        initial_dfc_activities_list = ultimo_dfc['ATIVIDADES'].tolist()

dates_to_check = []
current_date_iter = date.today()
while len(dates_to_check) < 4:
    if current_date_iter.weekday() < 5: dates_to_check.append(current_date_iter)
    current_date_iter += timedelta(days=1)

# --- 4. GERAÇÃO E EXPORTAÇÃO ---
generated_dfcs = {}
for d_date in dates_to_check:
    sheet_name = d_date.strftime('%Y-%m-%d')
    generated_dfcs[sheet_name] = generate_and_populate_dfc_for_date(
        d_date, initial_dfc_activities_list, despesa, RECEITA
    )
        
# --- 4. GERAÇÃO DAS NOVAS ABAS ---
# --- 3. TEMPLATE E DATAS (OTIMIZADO PARA AS ÚLTIMAS 7) ---
initial_dfc_activities_list = []
with pd.ExcelFile(excel_file_name) as xls:
    # Filtra apenas abas que parecem datas (formato YYYY-MM-DD)
    todas_abas = xls.sheet_names
    abas_dfc = [s for s in todas_abas if s not in ['BANCO DE DADOS', 'DESPESAS', 'RECEITA']]
    
    # PEGA APENAS AS ÚLTIMAS 7 PARA GANHAR VELOCIDADE
    abas_dfc_recentes = abas_dfc[-7:] if len(abas_dfc) > 7 else abas_dfc
    
    if abas_dfc_recentes:
        # Lê o template da última aba realmente relevante
        ultimo_dfc = pd.read_excel(xls, sheet_name=abas_dfc_recentes[-1])
        initial_dfc_activities_list = ultimo_dfc['ATIVIDADES'].tolist()

# ... (seu código de dates_to_check permanece igual)

# --- 4. GERAÇÃO E EXPORTAÇÃO ---
with pd.ExcelWriter(excel_file_name, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    for sheet_name, dfc_df in generated_dfcs.items():
        # Limpa números mantendo float
        cols_num = [col for col in dfc_df.columns if col != 'ATIVIDADES']
        dfc_export = clean_numeric_columns(dfc_df, cols_num)
        
        # CORREÇÃO DO ERRO DE FLOAT:
        # Identifica linhas vazias
        mask_vazio = (dfc_export['ATIVIDADES'].isna()) | (dfc_export['ATIVIDADES'].astype(str).str.strip() == '')
        
        for col in cols_num:
            # Converte a coluna para 'object' para permitir a mistura de número e string vazia
            dfc_export[col] = dfc_export[col].astype(object)
            dfc_export.loc[mask_vazio, col] = ''
        
        # Grava a aba
        dfc_export.to_excel(writer, sheet_name=sheet_name, index=False)
        
        # Formatação Visual (Quebra de texto e Moeda)
        ws = writer.sheets[sheet_name]
        for cell in ws[1]: # Linha 1: Cabeçalho
            cell.alignment = Alignment(wrapText=True, horizontal='center', vertical='center')
        
        # Formato de Moeda (Apenas onde há números)
        for row in ws.iter_rows(min_row=2, min_col=2):
            for cell in row:
                if isinstance(cell.value, (int, float)): # Só formata se for número
                    cell.number_format = '_-"R$"* #,##0.00_-'

# --- 5. REORDENAÇÃO AUTOMÁTICA (COLOCA AS NOVAS ANTES DAS BASES) ---
wb = openpyxl.load_workbook(excel_file_name)

# 1. Identifica todas as abas atuais
all_sheets = wb.sheetnames

# 2. Define as abas que DEVEM ser as últimas (as bases)
bases = ['DESPESAS', 'RECEITA']

# 3. Cria a nova ordem: todas as outras (incluindo as novas datas) vêm primeiro
# As bases vêm por último, não importa quantas abas novas foram criadas
nova_ordem = [s for s in all_sheets if s not in bases] + [s for s in bases if s in all_sheets]

# 4. Aplica a reordenação (isso é instantâneo e não afeta os dados)
wb._sheets = [wb[s] for s in nova_ordem]

# 5. Garante que o arquivo abra na aba 'DESPESAS' (mesmo ela estando no fim)
for sheet in wb.worksheets:
    sheet.views.sheetView[0].tabSelected = False

if 'DESPESAS' in wb.sheetnames:
    wb.active = wb.sheetnames.index('DESPESAS')
    wb['DESPESAS'].views.sheetView[0].tabSelected = True

wb.save(excel_file_name)
print("Sucesso! Novas planilhas inseridas antes de RECEITA e DESPESAS.")

# --- 5. ATIVAÇÃO DA ABA 'DESPESAS' NO FINAL ---
wb = openpyxl.load_workbook(excel_file_name)
for sheet in wb.worksheets:
    sheet.views.sheetView[0].tabSelected = False

if 'DESPESAS' in wb.sheetnames:
    despesa_idx = wb.sheetnames.index('DESPESAS')
    wb.active = despesa_idx
    wb['DESPESAS'].views.sheetView[0].tabSelected = True
    wb.save(excel_file_name)
    print("Sucesso! Bases no final e foco em DESPESAS.")

### **8. Assim que tiver atualizado o banco de dados, criar uma Posição entre 14d - se ainda estivermos antes dos 15d do mês - e 30d para após a metadade do mês.**

In [ ]:
# Get current date and time
now = datetime.now()
current_hour = now.hour
current_day = now.day
last_day_of_month = (now.replace(day=28) + timedelta(days=4)).day # Get last day of current month
try:
    # Determine date range based on hour and day of month
    date_range_days = 0
    if current_hour >= 17:
        if current_day <= 15:  # Mid-month
            date_range_days = 14
        elif current_day >= last_day_of_month - 7:  # Last week of the month
            date_range_days = 30

    print(f"Current hour: {current_hour}h")
    print(f"Current day of month: {current_day}")
    if date_range_days > 0:
        print(f"Determined date range: {date_range_days} days")
    else:
        print("No specific date range determined based on conditions (not 17h or not mid-month/last week).")


    # Helper functions for formatting (re-defined for self-containment)
    def format_currency(df, columns):
        """Formata colunas numéricas de um DataFrame para o formato monetário 'R$ X.XX'."""
        df_copy = df.copy()
        for col in columns:
            if col in df_copy.columns:
                df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce').fillna(0)
                df_copy[col] = df_copy[col].apply(lambda x: f'R$ {x:.2f}')
        return df_copy

    def format_dates(df, columns):
        """Formata colunas de data para o formato 'YYYY-MM-DD'."""
        df_copy = df.copy()
        for col in columns:
            if col in df_copy.columns:
                # O erro estava aqui: troque .datetime por .dt
                df_copy[col] = pd.to_datetime(df_copy[col], errors='coerce').dt.strftime('%Y-%m-%d')
        return df_copy

    # --- 1. Determine the range of dates for which to collect DFC sheets ---
    # Assuming 'today' is the date for which the DFC is being generated, or current date.
    # For this task, we will use the actual current date.
    today = date.today()

    # 'date_range_days' was determined in the previous step.
    # If it's still 0, we'll print a message and skip DFC collection.

    if date_range_days == 0:
        print("No specific DFC range was set for collection (date_range_days is 0). No DFC sheets will be collected for the new report.")
        dfc_sheets_for_export = []
    else:
        # Calculate start and end dates for DFC collection
        end_date = today
        start_date = today - timedelta(days=date_range_days - 1)

        print(f"Collecting DFC sheets from {start_date} to {end_date} (last {date_range_days} days).")

        excel_file_name = 'Relatório Financeiro 100k (1).xlsx'
        dfc_sheets_for_export = []

        try:
            # --- 2. Load all sheet names from the Relatório Financeiro 100k.xlsx file. ---
            xls = pd.ExcelFile(excel_file_name)
            all_sheet_names = xls.sheet_names

            # --- 3. Filter the loaded sheet names to identify those that match the 'YYYY-MM-DD' format and fall within the calculated date range. ---
            for sheet_name in all_sheet_names:
                if re.match(r'\d{4}-\d{2}-\d{2}', sheet_name):
                    try:
                        sheet_date = datetime.strptime(sheet_name, '%Y-%m-%d').date()
                        if start_date <= sheet_date <= end_date:
                            # --- 4. Load each identified DFC sheet into a separate DataFrame and store them in a list ---
                            dfc_sheets_for_export.append((sheet_date, pd.read_excel(excel_file_name, sheet_name=sheet_name)))
                    except ValueError:
                        # Not a valid date format, ignore
                        pass
        except FileNotFoundError:
            print(f"Error: The file '{excel_file_name}' was not found. No DFC sheets will be collected.")

        # --- 4. (continued) ensuring they are sorted by date. ---
        dfc_sheets_for_export.sort(key=lambda x: x[0])
        print(f"Collected {len(dfc_sheets_for_export)} DFC sheets within the specified date range.")

    RECEITA = pd.read_excel('Relatório Financeiro 100k (1).xlsx', sheet_name='RECEITA')
    despesa = pd.read_excel('Relatório Financeiro 100k (1).xlsx', sheet_name='DESPESAS')

    # --- 5. Prepare the RECEITA DataFrame for export by applying formatting ---
    receita_to_export = RECEITA.copy()
    currency_cols_receita = ['Valor da parcela', 'Valor Recebido', 'Valor Corrigido', 'Saldo', 'JUROS']
    date_cols_receita = ['Vencimento', 'Emissão', 'Data a considerar']
    receita_to_export = format_currency(receita_to_export, currency_cols_receita)
    receita_to_export = format_dates(receita_to_export, date_cols_receita)
    print("RECEITA DataFrame prepared with currency and date formatting.")

    # --- 6. Prepare the DESPESA DataFrame for export by applying formatting ---
    despesa_to_export = despesa.copy()
    currency_cols_despesa = ['VALOR PARCELA', 'JUROS', 'VALOR CORRIGIDO']
    date_cols_despesa = ['VENCIMENTO', 'EMISSÃO', 'DATA DE PAGAMENTO']
    despesa_to_export = format_currency(despesa_to_export, currency_cols_despesa)
    despesa_to_export = format_dates(despesa_to_export, date_cols_despesa)
    print("DESPESA DataFrame prepared with currency and date formatting.")

    timestamp = datetime.now().strftime('%d')
    output_excel_file_name = 'POSICAO_{}_FEV26.xlsx'.format(timestamp)

    # Create a new Excel file and write all sheets
    with pd.ExcelWriter(output_excel_file_name, engine='openpyxl') as writer:
        # Write DFC sheets if any were collected
        if dfc_sheets_for_export:
            for sheet_date, dfc_df in dfc_sheets_for_export:
                # Apply currency formatting to DFCs before writing, if not already done
                dfc_formatted_for_export_dfc = dfc_df.copy()
                all_numeric_cols_dfc = [col for col in dfc_df.columns if col not in ['ATIVIDADES']]
                dfc_formatted_for_export_dfc = format_currency(dfc_formatted_for_export_dfc, all_numeric_cols_dfc)

                # Ensure empty 'ATIVIDADES' rows are truly empty strings for export
                for col in dfc_formatted_for_export_dfc.columns:
                    if 'ATIVIDADES' in dfc_formatted_for_export_dfc.columns and \
                    dfc_formatted_for_export_dfc[dfc_formatted_for_export_dfc['ATIVIDADES'] == ''].shape[0] > 0:
                        dfc_formatted_for_export_dfc.loc[dfc_formatted_for_export_dfc['ATIVIDADES'] == '', col] = ''

                dfc_formatted_for_export_dfc.to_excel(writer, sheet_name=sheet_date.strftime('%Y-%m-%d'), index=False)
                print(f"Exported DFC for {sheet_date.strftime('%Y-%m-%d')}.")
        else:
            print("No DFC sheets to export.")

        # Write RECEITA and DESPESA DataFrames
        receita_to_export.to_excel(writer, sheet_name='RECEITA', index=False)
        despesa_to_export.to_excel(writer, sheet_name='DESPESAS', index=False)

    print(f"DataFrames exported successfully to '{output_excel_file_name}'.")

    # Set 'DESPESA' as the active sheet
    try:
        workbook = openpyxl.load_workbook(output_excel_file_name)
        despesa_sheet_index = -1
        for i, sheet_name in enumerate(workbook.sheetnames):
            if sheet_name == 'DESPESAS':
                despesa_sheet_index = i
                break

        if despesa_sheet_index != -1:
            workbook.active = workbook.worksheets[despesa_sheet_index]
            workbook.save(output_excel_file_name)
            print("\nPlanilha 'DESPESA' definida como a planilha ativa ao abrir o arquivo.")
        else:
            print("\nErro: A planilha 'DESPESA' não foi encontrada para ser definida como ativa.")

    except Exception as e:
        print(f"Ocorreu um erro ao definir a planilha ativa: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")